In [6]:
# Importing all necessary libraries

import pandas as pd
import numpy as np
import sys
import os
sys.path.insert(0, os.path.abspath('..')) 
from src.data_related.data_selector import *
from src.features.preprocessing import *
from src.models.trainers import *
from src.evaluation.leaderboards import *
from src.models.tuning import *
from src.visualisation.leaderboards_to_png import *

In [7]:
# Loading the dataframe
df = pd.read_csv('../datasets/processed/pkdx_min.csv', index_col='id')

# Reorganise the columns for easier reading and dropping names
tdf = df[['generation', 'type_1', 'type_2', 'rarity', 'stage', 'shape', 'color', 'height', 'weight', 'total_stats']].copy()


# Defining Gen10 test starters
new_gen_initials = pd.DataFrame({
    'name':['Browt', 'Pombon', 'Gecqua'],
    'generation': [10, 10, 10],
    'type_1': ['grass', 'fire', 'water'],
    'type_2': ['None', 'None', 'None'],
    'rarity': ['regular', 'regular', 'regular'],
    'stage': ['s1c3', 's1c3', 's1c3'],
    'shape': ['legs', 'quadruped', 'quadruped'],
    'color': ['yellow', 'red', 'blue'],
    'height': [0.3, 0.4, 0.3],
    'weight': [3.5, 6.7, 4.3]
})



common_params = {
    'learning_rate': np.arange(0.005, 0.035, 0.002),    
    'max_depth': np.arange(3, 10, 1),                     
    'n_estimators': np.arange(300, 3000, 50)            
}

# FULL ELEGANT GRID
params_grid = {
    'catboost': {
        **common_params,
        'random_strength': np.arange(2, 10, 1),          
        'bagging_temperature': np.arange(0.2, 1.6, 0.1),
        'l2_leaf_reg': np.arange(0.1, 2.5, 0.1),      
    },
    'light_gbm': {
        **common_params,

        'num_leaves': np.arange(10, 50, 5),            
        'min_child_samples': np.arange(20, 70, 5),    
        'reg_lambda': np.arange(1, 10, 1),             
        'bagging_fraction': np.arange(0.1, 1, 0.1),
    }
}      

In [8]:
data_ready = prepare_data(tdf, new_gen_initials)
tun = tuning(data_ready, params_grid, search_iter=3)
metrics_ld = metrics_leaderboard(data_ready, **tun)
stats_ld = stats_prediction(data_ready, **tun)



KeyboardInterrupt: 

In [ ]:
metrics_ld
leaderboard_to_png(metrics_ld.drop('Features Relevance', axis=1), color_up = "#910606", img_path='../figures/metrics_ld.png')

In [ ]:
stats_ld
leaderboard_to_png(stats_ld, color_up = "#0c2eb6", img_path='../figures/stats_ld.png')

In [ ]:
metrics_ld

In [ ]:
stats_ld

